## LangGraph Open Deep Research - Supervisor-Researcher Architecture

In this notebook, we'll explore the **supervisor-researcher delegation architecture** for conducting deep research with LangGraph.

You can visit this repository to see the original application: [Open Deep Research](https://github.com/langchain-ai/open_deep_research)

Let's jump in!

## What We're Building

This implementation uses a **hierarchical delegation pattern** where:

1. **User Clarification** - Optionally asks clarifying questions to understand the research scope
2. **Research Brief Generation** - Transforms user messages into a structured research brief
3. **Supervisor** - A lead researcher that analyzes the brief and delegates research tasks
4. **Parallel Researchers** - Multiple sub-agents that conduct focused research simultaneously
5. **Research Compression** - Each researcher synthesizes their findings
6. **Final Report** - All findings are combined into a comprehensive report

![Architecture Diagram](https://i.imgur.com/Q8HEZn0.png)

This differs from a section-based approach by allowing dynamic task decomposition based on the research question, rather than predefined sections.

---

# 🤝 Breakout Room #1
## Deep Research Foundations

In this breakout room, we'll understand the architecture and components of the Open Deep Research system.

## Task 1: Dependencies

You'll need API keys for Anthropic (for the LLM) and Tavily (for web search). We'll configure the system to use Anthropic's Claude Sonnet 4 exclusively.

In [1]:
import os
import getpass

os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ")
os.environ["TAVILY_API_KEY"] = getpass.getpass("Enter your Tavily API key: ")

## Task 2: State Definitions

The state structure is hierarchical with three levels:

### Agent State (Top Level)
Contains the overall conversation messages, research brief, accumulated notes, and final report.

### Supervisor State (Middle Level)
Manages the research supervisor's messages, research iterations, and coordinating parallel researchers.

### Researcher State (Bottom Level)
Each individual researcher has their own message history, tool call iterations, and research findings.

We also have structured outputs for tool calling:
- **ConductResearch** - Tool for supervisor to delegate research to a sub-agent
- **ResearchComplete** - Tool to signal research phase is done
- **ClarifyWithUser** - Structured output for asking clarifying questions
- **ResearchQuestion** - Structured output for the research brief

Let's import these from our library: [`open_deep_library/state.py`](open_deep_library/state.py)

In [2]:
# Import state definitions from the library
from open_deep_library.state import (
    # Main workflow states
    AgentState,           # Lines 65-72: Top-level agent state with messages, research_brief, notes, final_report
    AgentInputState,      # Lines 62-63: Input state is just messages
    
    # Supervisor states
    SupervisorState,      # Lines 74-81: Supervisor manages research delegation and iterations
    
    # Researcher states
    ResearcherState,      # Lines 83-90: Individual researcher with messages and tool iterations
    ResearcherOutputState, # Lines 92-96: Output from researcher (compressed research + raw notes)
    
    # Structured outputs for tool calling
    ConductResearch,      # Lines 15-19: Tool for delegating research to sub-agents
    ResearchComplete,     # Lines 21-22: Tool to signal research completion
    ClarifyWithUser,      # Lines 30-41: Structured output for user clarification
    ResearchQuestion,     # Lines 43-48: Structured output for research brief
)

## Task 3: Utility Functions and Tools

The system uses several key utilities:

### Search Tools
- **tavily_search** - Async web search with automatic summarization to stay within token limits
- Supports Anthropic native web search and Tavily API

### Reflection Tools
- **think_tool** - Allows researchers to reflect on their progress and plan next steps (ReAct pattern)

### Helper Utilities
- **get_all_tools** - Assembles the complete toolkit (search + MCP + reflection)
- **get_today_str** - Provides current date context for research
- Token limit handling utilities for graceful degradation

These are defined in [`open_deep_library/utils.py`](open_deep_library/utils.py)

In [3]:
# Import utility functions and tools from the library
from open_deep_library.utils import (
    # Search tool - Lines 43-136: Tavily search with automatic summarization
    tavily_search,
    
    # Reflection tool - Lines 219-244: Strategic thinking tool for ReAct pattern
    think_tool,
    
    # Tool assembly - Lines 569-597: Get all configured tools
    get_all_tools,
    
    # Date utility - Lines 872-879: Get formatted current date
    get_today_str,
    
    # Supporting utilities for error handling
    get_api_key_for_model,          # Lines 892-914: Get API keys from config or env
    is_token_limit_exceeded,         # Lines 665-701: Detect token limit errors
    get_model_token_limit,           # Lines 831-846: Look up model's token limit
    remove_up_to_last_ai_message,    # Lines 848-866: Truncate messages for retry
    anthropic_websearch_called,      # Lines 607-637: Detect Anthropic native search usage
    openai_websearch_called,         # Lines 639-658: Detect OpenAI native search usage
    get_notes_from_tool_calls,       # Lines 599-601: Extract notes from tool messages
)

## Task 4: Configuration System

The configuration system controls:

### Research Behavior
- **allow_clarification** - Whether to ask clarifying questions before research
- **max_concurrent_research_units** - How many parallel researchers can run (default: 5)
- **max_researcher_iterations** - How many times supervisor can delegate research (default: 6)
- **max_react_tool_calls** - Tool call limit per researcher (default: 10)

### Model Configuration
- **research_model** - Model for research and supervision (we'll use Anthropic)
- **compression_model** - Model for synthesizing findings
- **final_report_model** - Model for writing the final report
- **summarization_model** - Model for summarizing web search results

### Search Configuration
- **search_api** - Which search API to use (ANTHROPIC, TAVILY, or NONE)
- **max_content_length** - Character limit before summarization

Defined in [`open_deep_library/configuration.py`](open_deep_library/configuration.py)

In [4]:
# Import configuration from the library
from open_deep_library.configuration import (
    Configuration,    # Lines 38-247: Main configuration class with all settings
    SearchAPI,        # Lines 11-17: Enum for search API options (ANTHROPIC, TAVILY, NONE)
)

## Task 5: Prompt Templates

The system uses carefully engineered prompts for each phase:

### Phase 1: Clarification
**clarify_with_user_instructions** - Analyzes if the research scope is clear or needs clarification

### Phase 2: Research Brief
**transform_messages_into_research_topic_prompt** - Converts user messages into a detailed research brief

### Phase 3: Supervisor
**lead_researcher_prompt** - System prompt for the supervisor that manages delegation strategy

### Phase 4: Researcher
**research_system_prompt** - System prompt for individual researchers conducting focused research

### Phase 5: Compression
**compress_research_system_prompt** - Prompt for synthesizing research findings without losing information

### Phase 6: Final Report
**final_report_generation_prompt** - Comprehensive prompt for writing the final report

All prompts are defined in [`open_deep_library/prompts.py`](open_deep_library/prompts.py)

In [5]:
# Import prompt templates from the library
from open_deep_library.prompts import (
    clarify_with_user_instructions,                    # Lines 3-41: Ask clarifying questions
    transform_messages_into_research_topic_prompt,     # Lines 44-77: Generate research brief
    lead_researcher_prompt,                            # Lines 79-136: Supervisor system prompt
    research_system_prompt,                            # Lines 138-183: Researcher system prompt
    compress_research_system_prompt,                   # Lines 186-222: Research compression prompt
    final_report_generation_prompt,                    # Lines 228-308: Final report generation
)

## ❓ Question #1:

Explain the interrelationships between the three states (Agent, Supervisor, Researcher). Why don't we just make a single huge state?

##### Answer:
The three states represent different scopes of responsibility + different lifetimes of information:
1. AgentState (top-level) holds the end-to-end user journey: user messages, the research brief, accumulated notes, and the final report. It’s the “product-level state” that survives the entire workflow.

2. SupervisorState (middle-level) manages coordination: it receives the research brief, decides what sub-questions to investigate, spawns researcher tasks, and merges intermediate findings. It’s the “manager state” whose job is routing + quality control.

3. ResearcherState (bottom-level) is scoped to one research thread: a single research question, tool calls (search), raw notes, compression, and returning a clean summary back to the supervisor. It’s the “specialist state” optimized for short, focused context.

Why not one huge state?
- Separation of concerns: Mixing user convo + planning + tool logs + raw research across many sub-threads creates confusion and prompt drift.
- Token efficiency: Researcher work generates lots of noisy intermediate context (search snippets, partial notes). Keeping it isolated prevents bloating the main state.
- Parallelism: Multiple researchers need independent contexts to avoid cross-contamination.
- Debuggability: With layered states you can inspect failures at the right layer (researcher vs supervisor vs agent) instead of untangling one giant blob.

In short: the architecture makes state modular, bounded, and composable, which is exactly what “deep research” needs.

## ❓ Question #2:

What are the advantages and disadvantages of importing these components instead of including them in the notebook?

Advantages
- Reusability: State definitions, prompts, and nodes become reusable modules across notebooks/apps.
- Maintainability: Fix once in the library → all notebooks benefit.
- Clarity: Notebook focuses on learning + orchestration instead of 500 lines of plumbing.
- Testability: Library functions can be unit-tested (state schemas, tool wrappers, routing logic).
- Production alignment: Mirrors real-world design (codebase modules, not “notebook spaghetti”).

Disadvantages
- Reduced visibility for learning: Important logic becomes “hidden,” so it’s harder to understand without opening source files.
- Harder to customize quickly: Small modifications require editing library code or overriding config/prompt injection.
- Environment coupling: Imports assume correct package installation + compatible versions.
- Debug friction: Stack traces jump into library code; beginners may struggle to locate root cause.

Best practice: keep core architecture in the library, but expose configuration knobs + prompt overrides in the notebook so experimentation stays easy.

##### Answer:


## 🏗️ Activity #1: Explore the Prompts

Open `open_deep_library/prompts.py` and examine one of the prompt templates in detail.

**Requirements:**
1. Choose one prompt template (clarify, brief, supervisor, researcher, compression, or final report)
2. Explain what the prompt is designed to accomplish
3. Identify 2-3 key techniques used in the prompt (e.g., structured output, role definition, examples)
4. Suggest one improvement you might make to the prompt

**YOUR CODE HERE** - Write your analysis in a markdown cell below

1) Which prompt I chose
- I examined lead_researcher_prompt (the supervisor’s system prompt).

2) What it is designed to accomplish
This prompt is designed to turn the model into a research manager that:
- interprets the research brief,
- decomposes it into sub-questions,
- delegates to researcher subagents (or researcher subgraph runs),
- aggregates findings,
- and decides when there’s enough signal to produce the final report.
In other words: it’s optimizing for coordination, coverage, and quality, not raw knowledge.

3) 2–3 key techniques used

(a) Role + responsibility boundaries
The prompt explicitly frames the model as a lead researcher/supervisor, which reduces “do everything” behavior and encourages delegation + structured thinking.

(b) Structured outputs / explicit planning
Supervisor prompts typically enforce outputs like:
- list of sub-questions,
- assignments for researchers,
- expected deliverables,
- and a synthesis plan.
- This prevents rambling and improves traceability.

(c) Quality constraints / evidence posture
Most deep research supervisor prompts include constraints like:
- prefer credible sources,
- avoid hallucination,
- keep track of what’s known vs unknown,
- and ensure the final report is grounded in gathered notes.

4) One improvement I would make
- Add an explicit “coverage checklist + stop condition”:
- A short rubric like: “Do we have 3–5 credible sources per major claim? Have we covered counterpoints? Any open questions?”
- A stop rule like: “If research confidence ≥ X and todos are complete → proceed to final report.”
This improves reliability and prevents “keep researching forever” behavior.

---

# 🤝 Breakout Room #2
## Building & Running the Researcher

In this breakout room, we'll explore the node functions, build the graph, and run wellness research.

## Task 6: Node Functions - The Building Blocks

Now let's look at the node functions that make up our graph. We'll import them from the library and understand what each does.

### The Complete Research Workflow

The workflow consists of 8 key nodes organized into 3 subgraphs:

1. **Main Graph Nodes:**
   - `clarify_with_user` - Entry point that checks if clarification is needed
   - `write_research_brief` - Transforms user input into structured research brief
   - `final_report_generation` - Synthesizes all research into final report

2. **Supervisor Subgraph Nodes:**
   - `supervisor` - Lead researcher that plans and delegates
   - `supervisor_tools` - Executes supervisor's tool calls (delegation, reflection)

3. **Researcher Subgraph Nodes:**
   - `researcher` - Individual researcher conducting focused research
   - `researcher_tools` - Executes researcher's tool calls (search, reflection)
   - `compress_research` - Synthesizes researcher's findings

All nodes are defined in [`open_deep_library/deep_researcher.py`](open_deep_library/deep_researcher.py)

### Node 1: clarify_with_user

**Purpose:** Analyzes user messages and asks clarifying questions if the research scope is unclear.

**Key Steps:**
1. Check if clarification is enabled in configuration
2. Use structured output to analyze if clarification is needed
3. If needed, end with a clarifying question for the user
4. If not needed, proceed to research brief with verification message

**Implementation:** [`open_deep_library/deep_researcher.py` lines 60-115](open_deep_library/deep_researcher.py#L60-L115)

In [6]:
# Import the clarify_with_user node
from open_deep_library.deep_researcher import clarify_with_user

### Node 2: write_research_brief

**Purpose:** Transforms user messages into a structured research brief for the supervisor.

**Key Steps:**
1. Use structured output to generate detailed research brief from messages
2. Initialize supervisor with system prompt and research brief
3. Set up supervisor messages with proper context

**Why this matters:** A well-structured research brief helps the supervisor make better delegation decisions.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 118-175](open_deep_library/deep_researcher.py#L118-L175)

In [7]:
# Import the write_research_brief node
from open_deep_library.deep_researcher import write_research_brief

### Node 3: supervisor

**Purpose:** Lead research supervisor that plans research strategy and delegates to sub-researchers.

**Key Steps:**
1. Configure model with three tools:
   - `ConductResearch` - Delegate research to a sub-agent
   - `ResearchComplete` - Signal that research is done
   - `think_tool` - Strategic reflection before decisions
2. Generate response based on current context
3. Increment research iteration count
4. Proceed to tool execution

**Decision Making:** The supervisor uses `think_tool` to reflect before delegating research, ensuring thoughtful decomposition of the research question.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 178-223](open_deep_library/deep_researcher.py#L178-L223)

In [8]:
# Import the supervisor node (from supervisor subgraph)
from open_deep_library.deep_researcher import supervisor

### Node 4: supervisor_tools

**Purpose:** Executes the supervisor's tool calls, including strategic thinking and research delegation.

**Key Steps:**
1. Check exit conditions:
   - Exceeded maximum iterations
   - No tool calls made
   - `ResearchComplete` called
2. Process `think_tool` calls for strategic reflection
3. Execute `ConductResearch` calls in parallel:
   - Spawn researcher subgraphs for each delegation
   - Limit to `max_concurrent_research_units` (default: 5)
   - Gather all results asynchronously
4. Aggregate findings and return to supervisor

**Parallel Execution:** This is where the magic happens - multiple researchers work simultaneously on different aspects of the research question.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 225-349](open_deep_library/deep_researcher.py#L225-L349)

In [9]:
# Import the supervisor_tools node
from open_deep_library.deep_researcher import supervisor_tools

### Node 5: researcher

**Purpose:** Individual researcher that conducts focused research on a specific topic.

**Key Steps:**
1. Load all available tools (search, MCP, reflection)
2. Configure model with tools and researcher system prompt
3. Generate response with tool calls
4. Increment tool call iteration count

**ReAct Pattern:** Researchers use `think_tool` to reflect after each search, deciding whether to continue or provide their answer.

**Available Tools:**
- Search tools (Tavily or Anthropic native search)
- `think_tool` for strategic reflection
- `ResearchComplete` to signal completion
- MCP tools (if configured)

**Implementation:** [`open_deep_library/deep_researcher.py` lines 365-424](open_deep_library/deep_researcher.py#L365-L424)

In [10]:
# Import the researcher node (from researcher subgraph)
from open_deep_library.deep_researcher import researcher

### Node 6: researcher_tools

**Purpose:** Executes the researcher's tool calls, including searches and strategic reflection.

**Key Steps:**
1. Check early exit conditions (no tool calls, native search used)
2. Execute all tool calls in parallel:
   - Search tools fetch and summarize web content
   - `think_tool` records strategic reflections
   - MCP tools execute external integrations
3. Check late exit conditions:
   - Exceeded `max_react_tool_calls` (default: 10)
   - `ResearchComplete` called
4. Continue research loop or proceed to compression

**Error Handling:** Safely handles tool execution errors and continues with available results.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 435-509](open_deep_library/deep_researcher.py#L435-L509)

In [11]:
# Import the researcher_tools node
from open_deep_library.deep_researcher import researcher_tools

### Node 7: compress_research

**Purpose:** Compresses and synthesizes research findings into a concise, structured summary.

**Key Steps:**
1. Configure compression model
2. Add compression instruction to messages
3. Attempt compression with retry logic:
   - If token limit exceeded, remove older messages
   - Retry up to 3 times
4. Extract raw notes from tool and AI messages
5. Return compressed research and raw notes

**Why Compression?** Researchers may accumulate lots of tool outputs and reflections. Compression ensures:
- All important information is preserved
- Redundant information is deduplicated
- Content stays within token limits for the final report

**Token Limit Handling:** Gracefully handles token limit errors by progressively truncating messages.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 511-585](open_deep_library/deep_researcher.py#L511-L585)

In [12]:
# Import the compress_research node
from open_deep_library.deep_researcher import compress_research

### Node 8: final_report_generation

**Purpose:** Generates the final comprehensive research report from all collected findings.

**Key Steps:**
1. Extract all notes from completed research
2. Configure final report model
3. Attempt report generation with retry logic:
   - If token limit exceeded, truncate findings by 10%
   - Retry up to 3 times
4. Return final report or error message

**Token Limit Strategy:**
- First retry: Use model's token limit × 4 as character limit
- Subsequent retries: Reduce by 10% each time
- Graceful degradation with helpful error messages

**Report Quality:** The prompt guides the model to create well-structured reports with:
- Proper headings and sections
- Inline citations
- Comprehensive coverage of all findings
- Sources section at the end

**Implementation:** [`open_deep_library/deep_researcher.py` lines 607-697](open_deep_library/deep_researcher.py#L607-L697)

In [13]:
# Import the final_report_generation node
from open_deep_library.deep_researcher import final_report_generation

## Task 7: Graph Construction - Putting It All Together

The system is organized into three interconnected graphs:

### 1. Researcher Subgraph (Bottom Level)
Handles individual focused research on a specific topic:
```
START → researcher → researcher_tools → compress_research → END
               ↑            ↓
               └────────────┘ (loops until max iterations or ResearchComplete)
```

### 2. Supervisor Subgraph (Middle Level)
Manages research delegation and coordination:
```
START → supervisor → supervisor_tools → END
            ↑              ↓
            └──────────────┘ (loops until max iterations or ResearchComplete)
            
supervisor_tools spawns multiple researcher_subgraphs in parallel
```

### 3. Main Deep Researcher Graph (Top Level)
Orchestrates the complete research workflow:
```
START → clarify_with_user → write_research_brief → research_supervisor → final_report_generation → END
                 ↓                                       (supervisor_subgraph)
               (may end early if clarification needed)
```

Let's import the compiled graphs from the library.

In [14]:
# Import the pre-compiled graphs from the library
from open_deep_library.deep_researcher import (
    # Bottom level: Individual researcher workflow
    researcher_subgraph,    # Lines 588-605: researcher → researcher_tools → compress_research
    
    # Middle level: Supervisor coordination
    supervisor_subgraph,    # Lines 351-363: supervisor → supervisor_tools (spawns researchers)
    
    # Top level: Complete research workflow
    deep_researcher,        # Lines 699-719: Main graph with all phases
)

## Why This Architecture?

### Advantages of Supervisor-Researcher Delegation

1. **Dynamic Task Decomposition**
   - Unlike section-based approaches with predefined structure, the supervisor can break down research based on the actual question
   - Adapts to different types of research (comparisons, lists, deep dives, etc.)

2. **Parallel Execution**
   - Multiple researchers work simultaneously on different aspects
   - Much faster than sequential section processing
   - Configurable parallelism (1-20 concurrent researchers)

3. **ReAct Pattern for Quality**
   - Researchers use `think_tool` to reflect after each search
   - Prevents excessive searching and improves search quality
   - Natural stopping conditions based on information sufficiency

4. **Flexible Tool Integration**
   - Easy to add MCP tools for specialized research
   - Supports multiple search APIs (Anthropic, Tavily)
   - Each researcher can use different tool combinations

5. **Graceful Token Limit Handling**
   - Compression prevents token overflow
   - Progressive truncation in final report generation
   - Research can scale to arbitrary depths

### Trade-offs

- **Complexity:** More moving parts than section-based approach
- **Cost:** Parallel researchers use more tokens (but faster)
- **Unpredictability:** Research structure emerges dynamically

## Task 8: Running the Deep Researcher

Now let's see the system in action! We'll use it to research wellness strategies for improving sleep quality.

### Setup

We need to:
1. Set up the wellness research request
2. Configure the execution with Anthropic settings
3. Run the research workflow

In [15]:
# Set up the graph with Anthropic configuration
from IPython.display import Markdown, display
import uuid

# Note: deep_researcher is already compiled from the library
# For this demo, we'll use it directly without additional checkpointing
graph = deep_researcher

print("✓ Graph ready for execution")
print("  (Note: The graph is pre-compiled from the library)")

✓ Graph ready for execution
  (Note: The graph is pre-compiled from the library)


### Configuration for Anthropic

We'll configure the system to use:
- **Claude Sonnet 4** for all research, supervision, and report generation
- **Tavily** for web search (you can also use Anthropic's native search)
- **Moderate parallelism** (1 concurrent researcher for cost control)
- **Clarification enabled** (will ask if research scope is unclear)

In [16]:
# Configure for Anthropic with moderate settings
config = {
    "configurable": {
        # Model configuration - using Claude Sonnet 4 for everything
        "research_model": "anthropic:claude-sonnet-4-20250514",
        "research_model_max_tokens": 10000,
        
        "compression_model": "anthropic:claude-sonnet-4-20250514",
        "compression_model_max_tokens": 8192,
        
        "final_report_model": "anthropic:claude-sonnet-4-20250514",
        "final_report_model_max_tokens": 10000,
        
        "summarization_model": "anthropic:claude-sonnet-4-20250514",
        "summarization_model_max_tokens": 8192,
        
        # Research behavior
        "allow_clarification": True,
        "max_concurrent_research_units": 1,  # 1 parallel researcher
        "max_researcher_iterations": 2,      # Supervisor can delegate up to 2 times
        "max_react_tool_calls": 3,           # Each researcher can make up to 3 tool calls
        
        # Search configuration
        "search_api": "tavily",  # Using Tavily for web search
        "max_content_length": 50000,
        
        # Thread ID for this conversation
        "thread_id": str(uuid.uuid4())
    }
}

print("✓ Configuration ready")
print(f"  - Research Model: Claude Sonnet 4")
print(f"  - Max Concurrent Researchers: 1")
print(f"  - Max Iterations: 2")
print(f"  - Search API: Tavily")

✓ Configuration ready
  - Research Model: Claude Sonnet 4
  - Max Concurrent Researchers: 1
  - Max Iterations: 2
  - Search API: Tavily


### Execute the Wellness Research

Now let's run the research! We'll ask the system to research evidence-based strategies for improving sleep quality.

The workflow will:
1. **Clarify** - Check if the request is clear (may skip if obvious)
2. **Research Brief** - Transform our request into a structured brief
3. **Supervisor** - Plan research strategy and delegate to researchers
4. **Parallel Research** - Researchers gather information simultaneously
5. **Compression** - Each researcher synthesizes their findings
6. **Final Report** - All findings combined into comprehensive report

In [17]:
# Create our wellness research request
research_request = """
I want to improve my sleep quality. I currently:
- Go to bed at inconsistent times (10pm-1am)
- Use my phone in bed
- Often feel tired in the morning

Please research the best evidence-based strategies for improving sleep quality and create a comprehensive sleep improvement plan for me.
"""

# Execute the graph
async def run_research():
    """Run the research workflow and display results."""
    print("Starting research workflow...\n")
    
    async for event in graph.astream(
        {"messages": [{"role": "user", "content": research_request}]},
        config,
        stream_mode="updates"
    ):
        # Display each step
        for node_name, node_output in event.items():
            print(f"\n{'='*60}")
            print(f"Node: {node_name}")
            print(f"{'='*60}")
            
            if node_name == "clarify_with_user":
                if "messages" in node_output:
                    last_msg = node_output["messages"][-1]
                    print(f"\n{last_msg.content}")
            
            elif node_name == "write_research_brief":
                if "research_brief" in node_output:
                    print(f"\nResearch Brief Generated:")
                    print(f"{node_output['research_brief'][:500]}...")
            
            elif node_name == "supervisor":
                print(f"\nSupervisor planning research strategy...")
                if "supervisor_messages" in node_output:
                    last_msg = node_output["supervisor_messages"][-1]
                    if hasattr(last_msg, 'tool_calls') and last_msg.tool_calls:
                        print(f"Tool calls: {len(last_msg.tool_calls)}")
                        for tc in last_msg.tool_calls:
                            print(f"  - {tc['name']}")
            
            elif node_name == "supervisor_tools":
                print(f"\nExecuting supervisor's tool calls...")
                if "notes" in node_output:
                    print(f"Research notes collected: {len(node_output['notes'])}")
            
            elif node_name == "final_report_generation":
                if "final_report" in node_output:
                    print(f"\n" + "="*60)
                    print("FINAL REPORT GENERATED")
                    print("="*60 + "\n")
                    display(Markdown(node_output["final_report"]))
    
    print("\n" + "="*60)
    print("Research workflow completed!")
    print("="*60)

# Run the research
await run_research()

Starting research workflow...


Node: clarify_with_user

I have sufficient information to proceed with your sleep improvement research. I understand you're looking for evidence-based strategies to address your current sleep challenges: inconsistent bedtimes (10pm-1am range), phone use in bed, and morning fatigue. I will now research the best scientifically-backed sleep optimization techniques and create a comprehensive, personalized sleep improvement plan for you.

Node: write_research_brief

Research Brief Generated:
I want to improve my sleep quality using evidence-based strategies. My current sleep challenges include: going to bed at inconsistent times (ranging from 10pm to 1am), using my phone in bed, and often feeling tired in the morning despite sleeping. Please research the most effective, scientifically-backed sleep optimization techniques and create a comprehensive, personalized sleep improvement plan for me. The research should focus on: (1) evidence-based strategies for esta


Node: research_supervisor

Node: final_report_generation

FINAL REPORT GENERATED



# Evidence-Based Sleep Optimization Plan: Addressing Irregular Sleep Patterns and Morning Fatigue

Your sleep challenges—inconsistent bedtimes ranging from 10pm to 1am, phone use in bed, and persistent morning fatigue—are interconnected issues that can be effectively addressed through evidence-based interventions. Sleep research demonstrates that these problems create a cascade effect: irregular sleep timing disrupts your circadian rhythm, screen exposure compounds this disruption by suppressing melatonin production, and the resulting poor sleep quality leads to morning fatigue and continued cycle perpetuation.

## Establishing Consistent Sleep Schedules: The Foundation of Sleep Optimization

Circadian rhythm regulation is fundamental to improving sleep quality, and your 3-hour bedtime variation significantly disrupts this natural biological clock. Research from the Sleep Research Society demonstrates that even modest improvements in sleep timing consistency can yield substantial benefits within 2-3 weeks of implementation.

The most effective approach for establishing consistent sleep schedules involves gradual adjustment rather than abrupt changes. Sleep medicine research recommends shifting your bedtime by no more than 15-30 minutes earlier each night when trying to establish an earlier, more consistent schedule. Given your current 10pm-1am range, the optimal strategy is to identify a realistic target bedtime—likely around 11pm initially—and work systematically toward that goal.

Light exposure timing plays a crucial role in this process. Morning bright light exposure within 30 minutes of waking helps anchor your circadian rhythm and makes it easier to fall asleep at a consistent time each night. Research published in the Journal of Clinical Medicine shows that 20-30 minutes of bright light exposure (preferably natural sunlight or 10,000 lux artificial light) in the morning can advance sleep onset by 30-60 minutes within one week.

Weekend sleep schedule consistency is equally important. The phenomenon known as "social jet lag"—where weekend sleep schedules differ significantly from weekday patterns—can undo progress made during the week. Sleep research indicates that maintaining bedtime and wake time within one hour of your weekday schedule, even on weekends, significantly improves overall sleep quality and reduces Monday morning fatigue.

## Screen Time and Electronic Device Impact: The Blue Light Challenge

Your phone use in bed directly interferes with sleep through multiple mechanisms that extend beyond simple behavioral disruption. The blue light emitted by phone screens has a wavelength of approximately 460-480 nanometers, which most effectively suppresses melatonin production—the hormone responsible for promoting sleepiness.

Research from Harvard Medical School demonstrates that blue light exposure in the evening can suppress melatonin production by up to 50% and delay sleep onset by an average of 30-45 minutes. More concerning is that this effect can persist even after you stop using the device, meaning phone use right before sleep continues to impact your ability to fall asleep for 30-60 minutes afterward.

The most effective intervention is implementing a "digital sunset"—ceasing all screen use 1-2 hours before your target bedtime. However, recognizing that this may be challenging initially, research supports several graduated approaches. Blue light filtering glasses worn 2-3 hours before bedtime can reduce melatonin suppression by 60-70%, according to studies published in Chronobiology International. Similarly, enabling "night mode" or blue light filters on devices provides some benefit, though it's less effective than complete cessation.

Creating physical barriers to phone use in bed proves more effective than relying solely on willpower. Sleep hygiene research recommends charging phones outside the bedroom entirely, using a traditional alarm clock instead of phone alarms, and implementing a "phone parking station" in another room 1-2 hours before bedtime. If complete bedroom banishment isn't immediately feasible, keeping the phone at least arm's length away and face-down significantly reduces the temptation for middle-of-the-night usage.

## Improving Morning Alertness and Reducing Fatigue

Morning fatigue despite adequate sleep duration often indicates poor sleep quality rather than insufficient sleep quantity. This can result from frequent sleep stage disruptions, poor sleep environment conditions, or misalignment between your natural circadian rhythm and your wake time.

Strategic light exposure upon waking represents the most powerful tool for improving morning alertness. Research from the Journal of Sleep Research shows that bright light exposure immediately upon waking increases morning alertness scores by 40-60% within one week. This can be achieved through opening curtains immediately upon waking, using a dawn simulation alarm clock, or spending 10-15 minutes outdoors within 30 minutes of waking.

Sleep inertia—the groggy feeling upon waking—can be minimized through consistent wake times and avoiding the snooze button. Sleep research demonstrates that snoozing fragmentes the final sleep cycles and increases grogginess by up to 4 hours post-awakening. Setting your alarm for your actual intended wake time and getting up immediately, even if you feel tired initially, trains your body to anticipate awakening and reduces sleep inertia over time.

Temperature regulation also impacts morning alertness. Research indicates that keeping your bedroom slightly cool (65-68°F) during sleep and then allowing natural warming upon waking enhances the natural cortisol awakening response that promotes alertness. Some individuals benefit from programmable thermostats that slightly raise room temperature 30 minutes before wake time.

## Evidence-Based Sleep Hygiene Practices

Sleep hygiene encompasses environmental and behavioral factors that promote consistent, quality sleep. Research from the American Academy of Sleep Medicine identifies several core practices with robust evidence supporting their effectiveness.

The bedroom environment should be optimized for sleep through temperature, light, and noise control. Studies show that room temperatures between 65-68°F promote the deepest sleep by facilitating the natural core body temperature drop that occurs during sleep onset. Complete darkness is crucial—even small amounts of light can disrupt sleep architecture. Research published in Sleep Medicine Reviews demonstrates that blackout curtains, eye masks, or covering LED lights on electronics can improve sleep efficiency by 15-25%.

Noise management is equally important. Consistent background noise (white noise, fan, or earplugs) is more conducive to quality sleep than intermittent quiet periods interrupted by sudden sounds. Studies show that noise consistency is more important than absolute silence for most individuals.

Pre-sleep routines signal your body to prepare for sleep and can improve sleep onset time by 20-30 minutes when implemented consistently. Effective routines typically last 30-60 minutes and include relaxing activities such as reading, gentle stretching, meditation, or warm baths. The key is consistency—performing the same sequence of activities each night trains your body to associate these behaviors with sleep preparation.

Caffeine management significantly impacts sleep quality, with effects lasting much longer than most people realize. Research shows that caffeine consumed even 6 hours before bedtime can reduce sleep efficiency by 10-15%. Given your inconsistent bedtime schedule, avoiding caffeine after 2pm provides the safest margin to prevent sleep interference.

## Additional Evidence-Based Interventions for Irregular Sleep Patterns

Several advanced strategies can accelerate your transition to consistent, quality sleep, particularly given your specific pattern of irregular bedtimes and morning fatigue.

Melatonin supplementation, when properly timed, can help establish more consistent sleep patterns. Research from the Journal of Clinical Sleep Medicine shows that low-dose melatonin (0.5-3mg) taken 30-60 minutes before your desired bedtime can advance sleep onset and improve consistency. However, timing is crucial—taking melatonin at inconsistent times can worsen circadian rhythm disruption.

Sleep restriction therapy, a component of cognitive behavioral therapy for insomnia, can be particularly effective for individuals with irregular sleep patterns. This involves temporarily limiting time in bed to match your actual sleep time, then gradually increasing it as sleep efficiency improves. For someone with your pattern, this might initially mean going to bed later but waking at a consistent time, then gradually moving bedtime earlier as sleep quality improves.

Progressive muscle relaxation and mindfulness techniques address the mental arousal that often accompanies irregular sleep schedules. Research published in Sleep Medicine shows that 10-20 minutes of progressive muscle relaxation or mindfulness meditation before bed can improve sleep onset time by 25-40% and reduce middle-of-the-night awakenings.

Exercise timing impacts sleep quality, with morning or early afternoon exercise promoting better sleep than evening workouts. Studies indicate that regular aerobic exercise can improve sleep quality scores by 30-40%, but exercise within 4 hours of bedtime can delay sleep onset due to increased core body temperature and arousal.

## Implementation Strategy and Timeline

Given the interconnected nature of your sleep challenges, implementing changes gradually increases success likelihood. Week 1-2 should focus on establishing a consistent wake time and implementing morning light exposure, as these anchor points help stabilize your circadian rhythm. Week 3-4 can introduce the digital sunset and bedroom environment optimizations. Week 5-6 should integrate pre-sleep routines and fine-tune timing elements.

Tracking progress through sleep logs or apps can provide valuable feedback, but avoid becoming overly focused on metrics at the expense of overall consistency. Research shows that individuals who maintain sleep schedules within 30 minutes of their target times for 4-6 weeks experience significant improvements in sleep quality, morning alertness, and overall daytime functioning.

The key to success lies in treating these interventions as a comprehensive system rather than isolated changes. Your irregular bedtime pattern, phone use, and morning fatigue are interconnected problems requiring coordinated solutions. By addressing circadian rhythm stabilization, environmental optimization, and behavioral changes simultaneously, you can expect to see meaningful improvements in sleep quality within 2-4 weeks, with continued optimization over the following 2-3 months.

### Sources

[1] Sleep Research Society: Circadian Rhythm and Sleep Timing Guidelines: https://sleepresearchsociety.org/circadian-guidelines

[2] Journal of Clinical Medicine - Light Therapy for Sleep Disorders: https://www.mdpi.com/journal/jcm/special_issues/light_therapy_sleep

[3] Harvard Medical School - Blue Light and Sleep Research: https://www.health.harvard.edu/staying-healthy/blue-light-has-a-dark-side

[4] Chronobiology International - Blue Light Filtering Research: https://www.tandfonline.com/toc/icbi20/current

[5] Journal of Sleep Research - Morning Light Exposure Studies: https://onlinelibrary.wiley.com/journal/13652869

[6] American Academy of Sleep Medicine - Sleep Hygiene Guidelines: https://aasm.org/sleep-hygiene-tips

[7] Sleep Medicine Reviews - Environmental Sleep Factors: https://www.journals.elsevier.com/sleep-medicine-reviews

[8] Journal of Clinical Sleep Medicine - Melatonin Research: https://jcsm.aasm.org/


Research workflow completed!


## Task 9: Understanding the Output

Let's break down what happened:

### Phase 1: Clarification
The system checked if your request was clear. Since you provided specific details about your sleep issues, it likely proceeded without asking clarifying questions.

### Phase 2: Research Brief
Your request was transformed into a detailed research brief that guides the supervisor's delegation strategy.

### Phase 3: Supervisor Delegation
The supervisor analyzed the brief and decided how to break down the research:
- Used `think_tool` to plan strategy
- Called `ConductResearch` to delegate to researchers
- Each delegation specified a focused research topic (e.g., sleep hygiene, circadian rhythm, blue light effects)

### Phase 4: Parallel Research
Researchers worked on their assigned topics:
- Each researcher used web search tools to gather information
- Used `think_tool` to reflect after each search
- Decided when they had enough information
- Compressed their findings into clean summaries

### Phase 5: Final Report
All research findings were synthesized into a comprehensive sleep improvement plan with:
- Well-structured sections
- Evidence-based recommendations
- Practical action items
- Sources for further reading

## Task 10: Key Takeaways & Next Steps

### Architecture Benefits
1. **Dynamic Decomposition** - Research structure emerges from the question, not predefined
2. **Parallel Efficiency** - Multiple researchers work simultaneously
3. **ReAct Quality** - Strategic reflection improves search decisions
4. **Scalability** - Handles token limits gracefully through compression
5. **Flexibility** - Easy to add new tools and capabilities

### When to Use This Pattern
- **Complex research questions** that need multi-angle investigation
- **Comparison tasks** where parallel research on different topics is beneficial
- **Open-ended exploration** where structure should emerge dynamically
- **Time-sensitive research** where parallel execution speeds up results

### When to Use Section-Based Instead
- **Highly structured reports** with predefined format requirements
- **Template-based content** where sections are always the same
- **Sequential dependencies** where later sections depend on earlier ones
- **Budget constraints** where token efficiency is critical

### Extend the System
1. **Add MCP Tools** - Integrate specialized tools for your domain
2. **Custom Prompts** - Modify prompts for specific research types
3. **Different Models** - Try different Claude versions or mix models
4. **Persistence** - Use a real database for checkpointing instead of memory

### Learn More
- [LangGraph Documentation](https://langchain-ai.github.io/langgraph/)
- [Open Deep Research Repo](https://github.com/langchain-ai/open_deep_research)
- [Anthropic Claude Documentation](https://docs.anthropic.com/)
- [Tavily Search API](https://tavily.com/)

## ❓ Question #3:

What are the trade-offs of using parallel researchers vs. sequential research? When might you choose one approach over the other?

##### Answer:
Parallel researchers:
✅ Faster wall-clock time (multiple angles at once)
✅ Better breadth/coverage (different sub-questions explored simultaneously)
✅ Lower risk of tunnel vision (reduces single-thread bias)
❌ Higher cost (more model calls + more tool calls)
❌ Harder synthesis (must reconcile overlaps/conflicts)
❌ More context management overhead (merging notes, deduping)

Use parallel when:
- the topic has multiple subdomains,
- you need quick coverage,
- or you’re exploring unknown territory.

Sequential research
✅ Cheaper and simpler
✅ Each step can build on previous findings (better dependency handling)
✅ Easier to maintain a single narrative
❌ Slower
❌ Higher tunnel-vision risk (early assumptions steer everything)
❌ Can miss alternative angles

Use sequential when:
- the question is narrow,
- sources are limited,
- or steps are naturally dependent (A must be known before B).

## ❓ Question #4:

How would you adapt this deep research architecture for a production wellness application? What additional components would you need?

##### Answer:
To productionize this for wellness, I’d keep the supervisor-researcher architecture but add:

1. Safety + policy layer
- medical boundary rules (“informational, not diagnostic”)
- escalation paths (recommend clinician)
- red-flag detection (sleep apnea symptoms, severe depression markers)

2. User memory + personalization
- semantic profile (goals, constraints, preferences)
- episodic history (what worked last time)
- explicit consent and data retention controls

3. Source governance
- allowlist of trusted domains (NIH, CDC, NHS, peer-reviewed journals)
- citation requirement for claims
- ranking + deduplication of sources

4. Observability + evaluation
- tracing (LangSmith-style)
- feedback loops (thumbs up/down; “helpful?”)
- automated evals (factuality, safety, personalization)

5. Cost + performance controls
- limit concurrent researchers
- caching repeated searches
- compress/summarize at boundaries

6. UX workflow
- clarification step tuned for frictionless user experience
- outputs formatted as actionable plans (checklists, 7-day starter plan, etc.)
In production, “deep research” is not just a graph — it’s policy, memory, source control, and observability.

## 🏗️ Activity #2: Custom Wellness Research

Using what you've learned, run a custom wellness research task.

**Requirements:**
1. Create a wellness-related research question (exercise, nutrition, stress, etc.)
2. Modify the configuration for your use case
3. Run the research and analyze the output
4. Document what worked well and what could be improved

**Experiment ideas:**
- Research exercise routines for specific conditions (bad knee, lower back pain)
- Compare different stress management techniques
- Investigate nutrition strategies for specific goals
- Explore meditation and mindfulness research

**YOUR CODE HERE**

In [22]:
import uuid
from IPython.display import Markdown, display

research_request = """
Create a 4-week knee-friendly beginner plan for mild patellofemoral-type discomfort.

Constraints:
- Home workouts
- No jumping
- No deep squats
- Equipment: resistance bands + light dumbbells
- Time: 25 min weekdays, 45 min weekends

Output format:
- 5 key principles + common mistakes
- Warm-up + cooldown (8–10 min total)
- 4-week plan (weekly progression + example day templates)
- Safety notes + when to see a professional

Keep it concise. Use citations only if available from retrieved sources.
"""

cfg = {**config}
cfg.setdefault("configurable", {})
cfg["configurable"] = {**cfg["configurable"], "thread_id": str(uuid.uuid4())}

# Throughput controls (avoid TPM spikes)
cfg["configurable"]["max_concurrent_research_units"] = 1
cfg["configurable"]["max_research_loops"] = 1

# Token controls (if supported)
cfg["configurable"]["final_report_model_max_tokens"] = 2500
cfg["configurable"]["research_model_max_tokens"] = 3500
cfg["configurable"]["compression_model_max_tokens"] = 1500
cfg["configurable"]["max_search_results"] = 3

async def run_research():
    print("Starting research workflow...\n")

    async for event in graph.astream(
        {"messages": [{"role": "user", "content": research_request}]},
        cfg,
        stream_mode="updates"
    ):
        for node_name, node_output in event.items():
            print(f"\n{'='*60}")
            print(f"Node: {node_name}")
            print(f"{'='*60}")

            if node_name == "clarify_with_user":
                if "messages" in node_output and node_output["messages"]:
                    print("\n" + node_output["messages"][-1].content)

            elif node_name == "write_research_brief":
                if "research_brief" in node_output:
                    print("\nResearch Brief Generated (preview):")
                    print(node_output["research_brief"][:500] + "...")

            elif node_name in ["supervisor", "research_supervisor"]:
                print("\nSupervisor planning research strategy...")

            elif node_name in ["supervisor_tools", "research_supervisor_tools"]:
                print("\nExecuting tool calls...")
                if "notes" in node_output:
                    print(f"Research notes collected: {len(node_output['notes'])}")

            elif node_name == "final_report_generation":
                if "final_report" in node_output:
                    print(f"\n" + "="*60)
                    print("FINAL REPORT GENERATED")
                    print("="*60 + "\n")
                    display(Markdown(node_output["final_report"]))
                if "error" in node_output:
                    print("\nFINAL REPORT ERROR:")
                    print(node_output["error"])

    print("\n" + "="*60)
    print("Research workflow completed!")
    print("="*60)

await run_research()


Starting research workflow...


Node: clarify_with_user

I have all the information needed to create your knee-friendly beginner workout plan. I understand you need a 4-week progressive program for mild patellofemoral discomfort that includes: home workouts with resistance bands and light dumbbells, avoids jumping and deep squats, follows a 25-minute weekday/45-minute weekend schedule, and includes the specific components you've outlined (principles, warm-up/cooldown, weekly progression, and safety guidelines). I'll now begin researching and developing your comprehensive plan.

Node: write_research_brief

Research Brief Generated (preview):
I need a comprehensive 4-week knee-friendly beginner exercise plan specifically designed for mild patellofemoral-type discomfort. The plan must be suitable for home workouts with no jumping or deep squats, using only resistance bands and light dumbbells as equipment. The workout schedule should be 25 minutes on weekdays and 45 minutes on weekends. 


RateLimitError: Error code: 429 - {'type': 'error', 'error': {'type': 'rate_limit_error', 'message': "This request would exceed your organization's rate limit of 30,000 input tokens per minute (org: 89a47c9c-35e4-4c40-860f-00a2fe754c38, model: claude-sonnet-4-20250514). For details, refer to: https://docs.claude.com/en/api/rate-limits. You can see the response headers for current usage. Please reduce the prompt length or the maximum tokens requested, or try again later. You may also contact sales at https://www.anthropic.com/contact-sales to discuss your options for a rate limit increase."}, 'request_id': 'req_011CY14xF4YuTCyXX5tpFRBs'}

In [23]:
print(config)

{'configurable': {'research_model': 'anthropic:claude-sonnet-4-20250514', 'research_model_max_tokens': 10000, 'compression_model': 'anthropic:claude-sonnet-4-20250514', 'compression_model_max_tokens': 8192, 'final_report_model': 'anthropic:claude-sonnet-4-20250514', 'final_report_model_max_tokens': 10000, 'summarization_model': 'anthropic:claude-sonnet-4-20250514', 'summarization_model_max_tokens': 8192, 'allow_clarification': True, 'max_concurrent_research_units': 1, 'max_researcher_iterations': 2, 'max_react_tool_calls': 3, 'search_api': 'tavily', 'max_content_length': 50000, 'thread_id': '88280a9b-ea9f-4b05-8743-9d97edb8584d'}}


Why summarization timed out
- The summarization step timed out (60s), so the pipeline returned the original uncompressed content.
- That increased downstream prompt size and caused later stages to exceed token throughput limits.

Why final report failed
- The final report step hit a 429 rate_limit_error due to exceeding the organization’s input token-per-minute limit.
- This typically happens when intermediate context grows too large (especially when compression fails) and multiple heavy calls occur within the same minute.

Plan to improve and further analyze
- Reduced prompt size, lowered research concurrency and loops, and capped max tokens

In [24]:
import uuid

cfg = {"configurable": dict(config["configurable"])}  # copy

# Keep clarify on (as you want)
cfg["configurable"]["allow_clarification"] = True

# ✅ Reduce token budget (biggest win)
cfg["configurable"]["research_model_max_tokens"] = 3000
cfg["configurable"]["compression_model_max_tokens"] = 2000
cfg["configurable"]["summarization_model_max_tokens"] = 1500
cfg["configurable"]["final_report_model_max_tokens"] = 2500

# ✅ Reduce retrieval payload size
cfg["configurable"]["max_content_length"] = 12000   # was 50000 (too big)

# ✅ Keep throughput low
cfg["configurable"]["max_concurrent_research_units"] = 1
cfg["configurable"]["max_researcher_iterations"] = 1  # was 2

# Optional: fewer tool calls per react loop (less bloat)
cfg["configurable"]["max_react_tool_calls"] = 2      # was 3

# New thread each run
cfg["configurable"]["thread_id"] = str(uuid.uuid4())


In [26]:
import uuid
from IPython.display import Markdown, display

research_request = """
Create a 4-week knee-friendly beginner plan for mild patellofemoral-type discomfort.

Constraints:
- Home workouts
- No jumping
- No deep squats
- Equipment: resistance bands + light dumbbells
- Time: 25 min weekdays, 45 min weekends

Deliverables:
1) 5 key principles + common mistakes (concise)
2) Warm-up + cooldown (total 8–10 minutes)
3) 4-week progression (weekly goals + day templates)
4) Safety notes + when to see a professional

Keep the output concise and practical. Use citations only if returned by sources.
"""

# ---- tuned config to avoid summarization timeouts + TPM 429 ----
cfg = {"configurable": dict(config["configurable"])}
cfg["configurable"]["thread_id"] = str(uuid.uuid4())

cfg["configurable"]["research_model_max_tokens"] = 3000
cfg["configurable"]["compression_model_max_tokens"] = 2000
cfg["configurable"]["summarization_model_max_tokens"] = 1500
cfg["configurable"]["final_report_model_max_tokens"] = 2500

cfg["configurable"]["max_content_length"] = 12000
cfg["configurable"]["max_concurrent_research_units"] = 1
cfg["configurable"]["max_researcher_iterations"] = 1
cfg["configurable"]["max_react_tool_calls"] = 2

async def run_research():
    print("Starting research workflow...\n")

    async for event in graph.astream(
        {"messages": [{"role": "user", "content": research_request}]},
        cfg,
        stream_mode="updates"
    ):
        for node_name, node_output in event.items():
            print(f"\n{'='*60}")
            print(f"Node: {node_name}")
            print(f"{'='*60}")

            if node_name == "clarify_with_user":
                if "messages" in node_output and node_output["messages"]:
                    print("\n" + node_output["messages"][-1].content)

            elif node_name == "write_research_brief":
                if "research_brief" in node_output:
                    print("\nResearch Brief Generated (preview):")
                    print(node_output["research_brief"])

            elif node_name in ["supervisor", "research_supervisor"]:
                print("\nSupervisor planning research strategy...")

            elif node_name in ["supervisor_tools", "research_supervisor_tools"]:
                print("\nExecuting tool calls...")
                if "notes" in node_output:
                    try:
                        print(f"Research notes collected: {len(node_output['notes'])}")
                    except Exception:
                        pass

            elif node_name == "final_report_generation":
                if "final_report" in node_output:
                    print("\n" + "="*60)
                    print("FINAL REPORT GENERATED")
                    print("="*60 + "\n")
                    display(Markdown(node_output["final_report"]))
                elif "error" in node_output:
                    print("\nFINAL REPORT ERROR:\n", node_output["error"])

    print("\n" + "="*60)
    print("Research workflow completed!")
    print("="*60)

await run_research()


Starting research workflow...


Node: clarify_with_user

I have all the information needed to create your knee-friendly beginner workout plan. I understand you need a 4-week home workout program specifically designed for mild patellofemoral discomfort, with no jumping or deep squats, using only resistance bands and light dumbbells. The plan should include 25-minute weekday sessions and 45-minute weekend sessions, along with key principles, warm-up/cooldown routines, progressive weekly goals, and safety guidelines. I'll now begin researching and developing your comprehensive plan.

Node: write_research_brief

Research Brief Generated (preview):
I need a comprehensive 4-week knee-friendly beginner exercise plan specifically designed for mild patellofemoral-type discomfort that can be performed at home. The plan must exclude jumping movements and deep squats, and can only use resistance bands and light dumbbells as equipment. The workout schedule should accommodate 25-minute sessions on w

# Comprehensive 4-Week Knee-Friendly Exercise Plan for Patellofemoral Discomfort

## Key Principles for Knee-Safe Exercise with Patellofemoral Issues

### Core Exercise Principles

**1. Progressive Loading:** Begin with minimal resistance and gradually increase intensity over 4-6 weeks. The patellofemoral joint responds best to controlled, progressive strengthening that allows tissue adaptation without inflammatory flare-ups.

**2. Range of Motion Control:** Limit knee flexion to 0-60 degrees initially, avoiding deep knee bends that increase patellofemoral joint stress. Research shows that patellofemoral contact pressure increases significantly beyond 60 degrees of knee flexion.

**3. Multi-Plane Strengthening:** Focus on strengthening the entire kinetic chain including hip abductors, external rotators, and quadriceps. Weakness in hip stabilizers contributes to altered knee mechanics and increased patellofemoral stress.

**4. Pain Monitoring:** Use a 0-10 pain scale where exercise-induced discomfort should not exceed 3/10 during activity and should return to baseline within 2 hours post-exercise. Pain that persists or increases indicates excessive loading.

**5. Functional Movement Patterns:** Emphasize proper movement mechanics during daily activities through exercises that mimic real-world movements while maintaining proper knee alignment.

### Common Mistakes to Avoid

- **Ignoring hip weakness:** Focusing solely on quadriceps strengthening while neglecting hip abductor and external rotator weakness
- **Too much, too soon:** Rapidly increasing exercise intensity or volume without allowing proper tissue adaptation
- **Poor knee alignment:** Allowing knee valgus (inward collapse) during exercises, which increases patellofemoral stress
- **Exercising through significant pain:** Continuing exercises when pain exceeds 3/10 or doesn't resolve within 2 hours
- **Inconsistent progression:** Skipping sessions or advancing too quickly through exercise progressions

## Warm-Up and Cool-Down Routine (8-10 Minutes Total)

### Warm-Up Routine (4-5 Minutes)

**Gentle Movement Preparation:**
- **Ankle circles and calf raises (1 minute):** 10 circles each direction, followed by 15 calf raises to activate lower leg circulation
- **Marching in place (1 minute):** Gentle knee lifts to 45 degrees, focusing on controlled movement
- **Hip circles (1 minute):** Standing hip circles 10 each direction, followed by gentle leg swings front-to-back
- **Resistance band activation (2 minutes):** Light resistance band exercises including clamshells (15 reps) and mini-band side steps (10 steps each direction) to activate hip stabilizers

### Cool-Down Routine (4-5 Minutes)

**Flexibility and Recovery:**
- **Quadriceps stretch (1 minute):** Gentle standing quad stretch, 30 seconds each leg, avoiding excessive knee flexion
- **Hamstring stretch (1 minute):** Seated or supine hamstring stretch, 30 seconds each leg
- **Hip flexor stretch (1 minute):** Standing hip flexor stretch, 30 seconds each leg
- **IT band stretch (1 minute):** Standing IT band stretch against wall, 30 seconds each leg
- **Patellofemoral mobilization (1 minute):** Gentle seated knee extension/flexion within comfortable range, followed by patellar glides if comfortable

## 4-Week Progressive Program

### Week 1: Foundation and Assessment
**Goal:** Establish baseline strength, improve movement awareness, and reduce acute symptoms

**Weekday Sessions (25 minutes):**
- Warm-up: 4 minutes
- Exercise circuit: 17 minutes
  - Wall sits (0-30 degrees): 3 sets, 10-20 seconds
  - Resistance band hip abduction: 2 sets of 12
  - Straight leg raises: 2 sets of 10 each leg
  - Glute bridges: 2 sets of 12
  - Standing calf raises: 2 sets of 15
  - Light dumbbell heel raises: 2 sets of 10
- Cool-down: 4 minutes

**Weekend Sessions (45 minutes):**
- Extended warm-up: 6 minutes
- Comprehensive circuit: 34 minutes
  - All weekday exercises with additional set
  - Resistance band monster walks: 2 sets of 10 steps each direction
  - Modified lunges (shallow): 2 sets of 8 each leg
  - Side-lying leg lifts: 2 sets of 12 each side
  - Standing hip external rotation with band: 2 sets of 10 each leg
  - Seated leg extensions (partial range): 2 sets of 10 each leg
- Extended cool-down: 5 minutes

### Week 2: Strengthening Progression
**Goal:** Increase resistance and exercise complexity while maintaining pain-free movement

**Progression Criteria:** Advance only if Week 1 exercises can be completed without pain >3/10 and no delayed onset symptoms.

**Weekday Sessions (25 minutes):**
- Increase wall sit duration to 20-30 seconds
- Progress to moderate resistance band exercises
- Add light dumbbell (2-5 lbs) arm movements during stable exercises
- Introduce single-leg stance exercises (30 seconds each leg)

**Weekend Sessions (45 minutes):**
- All weekday progressions plus:
- Step-ups onto 4-6 inch platform: 2 sets of 8 each leg
- Resistance band squats (0-45 degrees): 2 sets of 10
- Lateral band walks: 2 sets of 12 steps each direction

### Week 3: Functional Integration
**Goal:** Integrate multi-plane movements and improve functional strength

**Weekday Sessions (25 minutes):**
- Progress wall sits to 30-45 seconds
- Increase resistance band tension
- Add dynamic movements with directional changes
- Include proprioceptive challenges (eyes closed during single-leg stance)

**Weekend Sessions (45 minutes):**
- All weekday progressions plus:
- Multi-directional step-ups: 2 sets of 6 each direction, each leg
- Resistance band diagonal patterns: 2 sets of 8 each direction
- Modified squat-to-chair: 2 sets of 12

### Week 4: Advanced Strengthening
**Goal:** Maximize strength gains and prepare for advanced exercise progression

**Weekday Sessions (25 minutes):**
- Wall sits progressed to 45-60 seconds
- Heavier resistance bands or light dumbbells (5-8 lbs)
- Complex movement patterns combining upper and lower body
- Dynamic balance challenges

**Weekend Sessions (45 minutes):**
- All weekday progressions plus:
- Lateral step-downs (controlled): 2 sets of 6 each leg
- Resistance band squats with arm movements: 2 sets of 12
- Single-leg deadlifts with light weight: 2 sets of 6 each leg

## Safety Guidelines and Professional Consultation Criteria

### Exercise Safety Parameters

**Pain Monitoring System:**
- **Green Zone (0-3/10 pain):** Safe to continue exercise with current intensity
- **Yellow Zone (4-6/10 pain):** Reduce intensity, modify exercise, or take rest day
- **Red Zone (7-10/10 pain):** Stop exercise immediately and consider professional consultation

**Daily Self-Assessment:**
- Morning stiffness duration (should not exceed 30 minutes)
- Pain levels during daily activities
- Swelling or warmth around the kneecap
- Response to previous day's exercise

### Red Flags Requiring Immediate Professional Consultation

**Seek immediate medical attention if experiencing:**
- Sudden onset of severe knee pain (>7/10)
- Significant swelling that develops rapidly
- Knee instability or giving way during normal activities
- Locking or catching sensations in the knee
- Inability to bear weight on the affected leg
- Signs of infection (fever, warmth, redness around knee)

**Schedule routine consultation if experiencing:**
- Persistent pain that doesn't improve after 2 weeks of conservative management
- Gradual worsening of symptoms despite following the program
- Inability to progress through the exercise program due to pain
- Development of compensatory movement patterns or pain in other joints
- Uncertainty about exercise form or program progression

### Exercise Modification Guidelines

**When to modify exercises:**
- If pain exceeds 3/10 during any movement
- If symptoms worsen 2+ hours post-exercise
- If morning stiffness increases in duration or intensity
- If swelling increases after exercise sessions

**Common modifications:**
- Reduce range of motion
- Decrease resistance or weight
- Reduce repetitions or sets
- Substitute closed-chain exercises for open-chain movements
- Add external support (wall, chair) during balance exercises

### Long-Term Management Strategies

**Beyond the 4-week program:**
- Continue strengthening exercises 3-4 times per week for maintenance
- Gradually introduce higher-level activities based on pain response
- Consider formal physical therapy evaluation for persistent symptoms
- Implement activity modification strategies for symptom management
- Regular reassessment of exercise technique and progression

This comprehensive program provides a structured, evidence-based approach to managing mild patellofemoral discomfort through progressive exercise while prioritizing safety and proper movement mechanics. The key to success lies in consistent adherence to the progression criteria, careful pain monitoring, and seeking professional guidance when indicated.

### Sources

[1] Patellofemoral Pain Syndrome Exercise Guidelines: https://www.jospt.org/doi/10.2519/jospt.2019.0302
[2] Progressive Loading Protocols for Knee Pain: https://bjsm.bmj.com/content/54/15/902
[3] Hip Strengthening for Patellofemoral Pain: https://www.apta.org/ptjournal/article/2018/98/4/285
[4] Exercise Prescription for Knee Disorders: https://journals.lww.com/acsm-csmr/Abstract/2020/19040/Exercise_Prescription_for_Patellofemoral_Pain.00006.aspx


Research workflow completed!


### Fix Applied
- Reduced max token budgets for research/compression/summarization/final report.
- Reduced max_content_length to limit retrieved payload size.
- Reduced max_researcher_iterations and max_react_tool_calls to constrain context growth.
This lowered total input tokens and prevented 429 errors while preserving workflow behavior.

### Analysis of final output and improvements done
A key takeaway is that agent design is as much about orchestration as intelligence. Explicit planning stages, bounded iterations, and controlled context size are essential for building reliable multi-stage research agents in production settings.
